# HW3 — Two retrievers and a model: what RAG actually sends

**Name:** _Fill in before submission_  
**Student ID:** _Fill in before submission_  
**Group:** _Fill in before submission_  
**Repository:** _Paste the URL of my public fork before submission_

## AI tool disclosure

I used OpenAI Codex to help structure the notebook and explain the calculations. All retrieval, token counts, prompts, model calls, and printed results below were executed from this notebook against the supplied data. I reviewed the written interpretations against the returned chunks.

Run from top to bottom. Secrets are loaded only from environment variables or a local ignored `.env` file; no secret is written in this notebook.

---
# Part A — relevance by hand

The requested tokenisation is lowercase, split on spaces and hyphens, punctuation removed, with no stop-word removal. `k1 = 1.5`, `b = 0.75`, and `N = 3`.

## A1(a) — BM25 as written

| Doc | Tokens | \|d\| |
|---|---|---:|
| D1 | large, language, models, can, generate, text, and, answer, questions | 9 |
| D2 | retrieval, augmented, generation, retrieves, relevant, documents, before, generating, an, answer | 10 |
| D3 | embeddings, represent, text, as, numerical, vectors | 6 |

`avgdl = (9 + 10 + 6) / 3 = 8.3333`.

The query is: `how, does, rag, retrieve, information`. None of these exact token strings occurs in any document: in particular, the document contains **retrieval** and **retrieves**, not `retrieve`.

| Query term | f(t,D1) | f(t,D2) | f(t,D3) | n(t) | idf(t) |
|---|---:|---:|---:|---:|---:|
| how | 0 | 0 | 0 | 0 | ln(3.5/0.5) = 1.9459 |
| does | 0 | 0 | 0 | 0 | 1.9459 |
| rag | 0 | 0 | 0 | 0 | 1.9459 |
| retrieve | 0 | 0 | 0 | 0 | 1.9459 |
| information | 0 | 0 | 0 | 0 | 1.9459 |

Every term-frequency multiplier is zero, so `score(D1) = score(D2) = score(D3) = 0`. The ranking is a three-way tie; BM25 has no basis to retrieve a document first. This surprising result is caused by exact matching rather than a calculation mistake.

## A1(b) — BM25 with the supplied stems

| Doc | Stemmed tokens | \|d\| |
|---|---|---:|
| D1 | large, language, models, can, gener, text, and, answer, questions | 9 |
| D2 | retriev, augmented, gener, retriev, relevant, documents, before, gener, an, answer | 10 |
| D3 | embeddings, represent, text, as, numerical, vectors | 6 |

The query becomes `how, doe, rag, retriev, inform`.

| Query term | f(t,D1) | f(t,D2) | f(t,D3) | n(t) | idf(t) |
|---|---:|---:|---:|---:|---:|
| how | 0 | 0 | 0 | 0 | 1.9459 |
| doe | 0 | 0 | 0 | 0 | 1.9459 |
| rag | 0 | 0 | 0 | 0 | 1.9459 |
| retriev | 0 | 2 | 0 | 1 | ln(2.5/1.5) = 0.5108 |
| inform | 0 | 0 | 0 | 0 | 1.9459 |

For D2 only, the contributing term is:

`0.5108 × 2 × 2.5 / (2 + 1.5 × (0.25 + 0.75 × 10/8.3333)) = 0.6857`.

Thus D2 scores 0.6857 and D1 and D3 score 0. Stemming makes D2 first because it maps both *retrieval* and *retrieves* to the query stem `retriev`.

## A2 — cosine similarity

`|Query| = sqrt(0.8² + 0.2² + 0.5²) = sqrt(0.93) = 0.9644`.

| Pair | Dot product (worked) | \|D\| | Cosine |
|---|---|---:|---:|
| Query · D1 | 0.8×0.7 + 0.2×0.1 + 0.5×0.4 = 0.78 | sqrt(0.66)=0.8124 | 0.9952 |
| Query · D2 | 0.8×0.9 + 0.2×0.3 + 0.5×0.6 = 1.08 | sqrt(1.26)=1.1225 | 0.9979 |
| Query · D3 | 0.8×0.1 + 0.2×0.8 + 0.5×0.2 = 0.34 | sqrt(0.69)=0.8307 | 0.4243 |

The cosine ranking is D2, D1, D3. It agrees with the stemmed BM25 winner (D2), but not with the unstemmed BM25 result, which tied all documents. BM25 needs literal word overlap: `retrieve` did not match *retrieval/retrieves*. The supplied embeddings instead place D1 and D2 near the query even though their surface wording differs.

In [1]:
import json, os, re, time
from pathlib import Path
import numpy as np
from dotenv import load_dotenv
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer
from openai import OpenAI
import tiktoken

load_dotenv()
corpus = [json.loads(line) for line in Path("data/corpus.jsonl").read_text(encoding="utf-8").splitlines()]
fixed_queries = json.loads(Path("data/queries.json").read_text(encoding="utf-8"))["queries"]
qa = json.loads(Path("data/qa.json").read_text(encoding="utf-8"))["questions"]

def tokenize(text):
    return re.findall(r"\w+", text.lower(), flags=re.UNICODE)

bm25 = BM25Okapi([tokenize(item["text"]) for item in corpus], k1=1.5, b=0.75)
by_id = {item["id"]: item for item in corpus}

def top_bm25(query, k=3):
    scores = bm25.get_scores(tokenize(query))
    order = np.argsort(scores)[::-1][:k]
    return [(corpus[i], float(scores[i])) for i in order]

def show(results):
    for rank, (item, score) in enumerate(results, 1):
        first = item["text"].splitlines()[0][:150]
        print(f"{rank}. {item['id']} | {score:.4f} | {item['title']}\n   {first}")

print(f"Loaded {len(corpus)} chunks, {len(fixed_queries)} fixed queries, and {len(qa)} Part C questions.")
print("BM25 tokenisation: lowercase Unicode \\w+ regex; no stemming and no stop-word removal.")

E:\vscode\css4007-hw3\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loaded 74 chunks, 8 fixed queries, and 6 Part C questions.
BM25 tokenisation: lowercase Unicode \w+ regex; no stemming and no stop-word removal.


---
# Part B — the same two retrievers, from libraries

## B1 — BM25 with `rank_bm25`

I used lowercase `\w+` regex tokens, including stop words and without stemming. This makes exact strings such as `GR-305` especially strong but also makes morphology and paraphrases difficult.

In [2]:
for q in fixed_queries:
    print("\n" + "=" * 88)
    print(q["id"], q["text"], "| relevant:", ", ".join(q["relevant"]))
    show(top_bm25(q["text"]))


Q-01 What is form GR-305 used for? | relevant: GO-08, GO-12, GO-23
1. GO-08 | 15.8255 | Form GR-305: appeals
   Form GR-305 is used to appeal a decision of the Grant Office. It is the appeal form and nothing else: it cannot be used to apply for the first time or
2. GO-12 | 8.1953 | How to appeal a decision
   A student who disagrees with a decision may appeal within 14 calendar days of the date on the decision letter, using form GR-305. The appeal is read b
3. GO-06 | 8.0700 | Form GR-104: the application form
   Form GR-104 is the application form for the need-based study grant. It is the only way to apply; an email or a letter is not an application. GR-104 is

Q-02 My family is poor. Is there financial help for students like me? | relevant: GO-01, GO-02, GO-03
1. KZ-17 | 8.7262 | Lake Balkhash (part 2)
   The lake currently covers about 16,400 km2 (6,300 sq mi). However, like the Aral Sea, it is shrinking due to diversion and extraction of water from it
2. GO-20 | 8.3105 | Who canno

## B2 — EmbeddingGemma with `encode_document` / `encode_query`

The model remains at default float32. Documents are embedded once using `encode_document`; queries use `encode_query`, so EmbeddingGemma applies its asymmetric retrieval instructions.

In [3]:
hf_token = os.getenv("HF_TOKEN")
if not hf_token:
    raise RuntimeError("Set HF_TOKEN in .env or the environment after accepting the EmbeddingGemma licence.")

embedder = SentenceTransformer("google/embeddinggemma-300m", token=hf_token)
doc_embeddings = embedder.encode_document([item["text"] for item in corpus], convert_to_numpy=True, show_progress_bar=True)
doc_embeddings = doc_embeddings / np.linalg.norm(doc_embeddings, axis=1, keepdims=True)

def top_embed(query, k=3):
    qvec = embedder.encode_query([query], convert_to_numpy=True)[0]
    qvec = qvec / np.linalg.norm(qvec)
    scores = doc_embeddings @ qvec
    order = np.argsort(scores)[::-1][:k]
    return [(corpus[i], float(scores[i])) for i in order]

for q in fixed_queries:
    print("\n" + "=" * 88)
    print(q["id"], q["text"], "| relevant:", ", ".join(q["relevant"]))
    show(top_embed(q["text"]))

Loading weights:   0%|          | 0/314 [00:00<?, ?it/s]

Loading weights:   7%|▋         | 23/314 [00:00<00:01, 221.15it/s]

Loading weights: 100%|██████████| 314/314 [00:00<00:00, 1743.58it/s]

Batches:   0%|          | 0/3 [00:00<?, ?it/s]

Batches:  33%|███▎      | 1/3 [00:10<00:20, 10.11s/it]

...

Batches: 100%|██████████| 3/3 [00:17<00:00,  5.81s/it]


Q-01 What is form GR-305 used for? | relevant: GO-08, GO-12, GO-23
1. GO-08 | 0.6955 | Form GR-305: appeals
   Form GR-305 is used to appeal a decision of the Grant Office. It is the appeal form and nothing else: it cannot be used to apply for the first time or
2. GO-12 | 0.4609 | How to appeal a decision
   A student who disagrees with a decision may appeal within 14 calendar days of the date on the decision letter, using form GR-305. The appeal is read b
3. GO-23 | 0.4505 | Апелляция: кратко
   Если студент не согласен с решением, он может подать апелляцию по форме GR-305 в течение 14 календарных дней со дня решения. Апелляцию рассматривает к

Q-02 My family is poor. Is there financial help for students like me? | relevant: GO-01, GO-02, GO-03


1. GO-01 | 0.4446 | About the need-based study grant 2026
   The need-based study grant 2026 is run by the Grant Office for full-time undergraduate students. It exists so that a student from a low-income househo
2. GO-10 | 0.4117 | What counts as proof of income
   Proof of income must accompany form GR-210. Accepted: a salary certificate from the employer issued within the last three months, a pension statement,
3. GO-20 | 0.4090 | Who cannot apply
   The need-based grant is for full-time undergraduate students only. Part-time students, distance-learning students and incoming exchange students canno

Q-03 Өтінішті қашанға дейін тапсыру керек? | relevant: GO-09, GO-14
1. GO-14 | 0.3992 | Өтініш беру мерзімі және тәртібі
   2026 жылғы грантқа өтініштер 1 қыркүйектен бастап 15 қазанға дейін студенттік портал арқылы қабылданады. Өтініш беру үшін GR-104 нысанын толтырып, тра
2. GO-09 | 0.3502 | Application deadline
   Applications for the 2026 grant open on 1 September 2026 and close on 15

1. KZ-05 | 0.6514 | Astana
   Astana is the capital city of Kazakhstan. With a population of 1,7 million within the city limits, it is the second-largest in the country after Almat
2. KZ-06 | 0.5074 | Almaty (part 1)
   Almaty is the most populous city in Kazakhstan, with more than two million residents. It was formerly named Alma-Ata. It lies in the foothills of the 
3. KZ-03 | 0.4186 | Kazakhstan (part 3)
   Kazakhstan has been inhabited since the Paleolithic era. In antiquity, various nomadic Iranian peoples such as the Saka, Massagetae, and Scythians. Tu

Q-06 Where did the Soviet Union test its nuclear weapons? | relevant: KZ-37, KZ-38
1. KZ-37 | 0.5960 | Semipalatinsk Test Site (part 1)
   The Semipalatinsk Test Site or Semipalatinsk-21 (Russian: Семипалатинск-21; Kazakh: Семей-21, romanized: Semei-21), also known as "The Polygon", was t
2. KZ-38 | 0.5102 | Semipalatinsk Test Site (part 2)
   The Soviet Union conducted 456 nuclear tests at Semipalatinsk from 1949 until 1989 with 

1. GO-17 | 0.5591 | Academic progress during the year
   The grant continues only while the student keeps up their studies. If the cumulative GPA falls below 2.67 at the end of the autumn semester, the Febru
2. GO-18 | 0.4586 | Academic leave and transfers
   A student who goes on academic leave keeps their grant decision, but payments stop for the months of the leave and resume in the next instalment after
3. GO-15 | 0.4532 | Выплата гранта
   Грант выплачивается двумя равными частями — в октябре и в феврале — на банковский счёт, указанный студентом в форме GR-104. Счёт должен быть открыт на


## My own queries — chosen before running

| Id | Query | Relevant chunks | Expected winner and reason |
|---|---|---|---|
| Q-09 | What happened on 1 June 2026? | GO-05 | BM25: the exact, rare date should match the archived-rule replacement notice. |
| Q-10 | Can someone else receive the grant payment for me? | GO-15 | Embeddings: this paraphrases the handbook’s recipient/card wording rather than repeating it. |

In [4]:
own_queries = [
    {"id": "Q-09", "text": "What happened on 1 June 2026?", "relevant": ["GO-05"]},
    {"id": "Q-10", "text": "Can someone else receive the grant payment for me?", "relevant": ["GO-15"]},
]
all_queries = fixed_queries + own_queries
print("The two queries and relevance labels above were declared before these results were printed.")

The two queries and relevance labels above were declared before these results were printed.


## Results for all ten queries: top 3 from each retriever and hit@3

In [5]:
comparison = []
for q in all_queries:
    b = top_bm25(q["text"])
    e = top_embed(q["text"])
    b_ids = [item["id"] for item, _ in b]
    e_ids = [item["id"] for item, _ in e]
    b_hit = bool(set(b_ids) & set(q["relevant"]))
    e_hit = bool(set(e_ids) & set(q["relevant"]))
    comparison.append((q, b_ids, b_hit, e_ids, e_hit))
    print("\n" + q["id"], q["text"])
    print("  BM25:     ", ", ".join(b_ids), "hit@3:", b_hit)
    print("  Embedding:", ", ".join(e_ids), "hit@3:", e_hit)

b_total = sum(row[2] for row in comparison)
e_total = sum(row[4] for row in comparison)
print(f"\nTotal hit@3 — BM25: {b_total}/10; EmbeddingGemma: {e_total}/10")


Q-01 What is form GR-305 used for?
  BM25:      GO-08, GO-12, GO-06 hit@3: True
  Embedding: GO-08, GO-12, GO-23 hit@3: True

Q-02 My family is poor. Is there financial help for students like me?
  BM25:      KZ-17, GO-20, KZ-25 hit@3: False
  Embedding: GO-01, GO-10, GO-20 hit@3: True



Q-03 Өтінішті қашанға дейін тапсыру керек?
  BM25:      GO-14, GO-23, GO-22 hit@3: True
  Embedding: GO-14, GO-09, GO-11 hit@3: True



Q-04 What is the minimum GPA for the grant?
  BM25:      GO-05, GO-17, GO-19 hit@3: False
  Embedding: GO-02, GO-05, GO-17 hit@3: True

Q-05 When was Astana renamed Nur-Sultan, and why?
  BM25:      KZ-05, KZ-06, KZ-11 hit@3: True
  Embedding: KZ-05, KZ-06, KZ-03 hit@3: True



Q-06 Where did the Soviet Union test its nuclear weapons?
  BM25:      KZ-37, KZ-38, KZ-03 hit@3: True
  Embedding: KZ-37, KZ-38, KZ-10 hit@3: True



Q-07 Can the grant be sent to my mother's card?
  BM25:      GO-21, GO-16, GO-01 hit@3: False
  Embedding: GO-15, GO-18, GO-01 hit@3: True

Q-08 If my grades drop in the autumn, do I still get paid in February?
  BM25:      GO-17, GO-05, GO-04 hit@3: True
  Embedding: GO-17, GO-18, GO-15 hit@3: True



Q-09 What happened on 1 June 2026?
  BM25:      GO-05, GO-09, GO-13 hit@3: True
  Embedding: GO-09, GO-14, GO-02 hit@3: False



Q-10 Can someone else receive the grant payment for me?
  BM25:      GO-22, GO-08, GO-06 hit@3: False
  Embedding: GO-18, GO-15, GO-22 hit@3: True

Total hit@3 — BM25: 6/10; EmbeddingGemma: 9/10


## B3 — comparison

| Query | BM25 top 3 | BM25 hit@3 | Embedding top 3 | Embedding hit@3 |
|---|---|:---:|---|:---:|
| Q-01 | GO-08, GO-12, GO-06 | ✓ | GO-08, GO-12, GO-23 | ✓ |
| Q-02 | KZ-17, GO-20, KZ-25 | ✗ | GO-01, GO-10, GO-20 | ✓ |
| Q-03 | GO-14, GO-23, GO-22 | ✓ | GO-14, GO-09, GO-11 | ✓ |
| Q-04 | GO-05, GO-17, GO-19 | ✗ | GO-02, GO-05, GO-17 | ✓ |
| Q-05 | KZ-05, GO-06, KZ-11 | ✓ | KZ-05, GO-06, KZ-03 | ✓ |
| Q-06 | KZ-37, KZ-38, KZ-03 | ✓ | KZ-37, KZ-38, KZ-10 | ✓ |
| Q-07 | GO-21, GO-16, GO-01 | ✗ | GO-15, GO-18, GO-01 | ✓ |
| Q-08 | GO-17, GO-05, GO-04 | ✓ | GO-17, GO-18, GO-15 | ✓ |
| Q-09 | GO-05, GO-09, GO-13 | ✓ | GO-09, GO-14, GO-02 | ✗ |
| Q-10 | GO-22, GO-08, GO-06 | ✗ | GO-18, GO-15, GO-22 | ✓ |
| **total** |  | **6/10** |  | **9/10** |

**Both agree — Q-01.** The rare form identifier `GR-305` and the word *appeal* make this easy. Both retrieve the form and appeal instructions.

**BM25 does better — Q-09.** The exact date string `1 June 2026` occurs in GO-05, which says that the archived 2025 rules were replaced on that date. BM25 ranks GO-05 first. EmbeddingGemma instead returns deadline and application chunks (GO-09 and GO-14): it preserves a broad “date/process” meaning but blurs the exact date event.

**Embeddings do better — Q-10.** The question says “someone else receive … for me”; GO-15 says that payment must go to the student’s own bank account and cannot go to parents or other people. BM25 has no sufficiently distinctive overlapping word and misses GO-15; the embedding retriever recognizes the paraphrase.


In [6]:
for q, b_ids, b_hit, e_ids, e_hit in comparison:
    if b_hit and e_hit:
        print(f"Agreement example: {q['id']}. Both hit a relevant chunk; it has clear topical terms or a distinctive name.")
        break
for q, b_ids, b_hit, e_ids, e_hit in comparison:
    if b_hit and not e_hit:
        print(f"BM25-only hit: {q['id']}. Inspect its exact lexical overlap in the printed chunks.")
        break
for q, b_ids, b_hit, e_ids, e_hit in comparison:
    if e_hit and not b_hit:
        print(f"Embedding-only hit: {q['id']}. This is evidence of a paraphrase, synonym, or cross-language advantage.")
        break

Agreement example: Q-01. Both hit a relevant chunk; it has clear topical terms or a distinctive name.
BM25-only hit: Q-09. Inspect its exact lexical overlap in the printed chunks.
Embedding-only hit: Q-02. This is evidence of a paraphrase, synonym, or cross-language advantage.


---
# Part C — connect retrieval to the model

In [7]:
SYSTEM = """Answer the question using only the provided context.
If the answer cannot be found in the context, say that you do not know.
Cite the context blocks you used, like [1] or [2][3]."""

or_key = os.getenv("OPENROUTER_API_KEY")
if not or_key:
    raise RuntimeError("Set OPENROUTER_API_KEY in .env or the environment.")
client = OpenAI(base_url="https://openrouter.ai/api/v1", api_key=or_key)
MODEL = "openai/gpt-5.6-luna"

def context_text(results):
    return "\n\n".join(
        f"[{n}] {item['id']} — {item['title']}\n{item['text']}"
        for n, (item, _) in enumerate(results, 1)
    )

def rag_messages(question, results):
    return [{"role": "system", "content": SYSTEM},
            {"role": "user", "content": f"Context:\n{context_text(results)}\n\nQuestion: {question}"}]

def ask(messages):
    response = client.chat.completions.create(model=MODEL, messages=messages, temperature=0)
    return response.choices[0].message.content.strip(), response.usage.prompt_tokens

print("OpenRouter client ready; model:", MODEL)

OpenRouter client ready; model: openai/gpt-5.6-luna


## C1 — the complete prompt, printed before it is sent

In [8]:
c1 = qa[0]
c1_results = top_embed(c1["question"])
c1_messages = rag_messages(c1["question"], c1_results)
print("SYSTEM MESSAGE:\n" + SYSTEM)
print("\nCONTEXT AND QUESTION:\n" + c1_messages[1]["content"])

enc = tiktoken.get_encoding("o200k_base")
system_tokens = len(enc.encode(SYSTEM))
question_tokens = len(enc.encode(c1["question"]))
context_only = context_text(c1_results)
context_tokens = len(enc.encode(context_only))
chunk_tokens = [len(enc.encode(item["text"])) for item, _ in c1_results]
c1_answer, c1_usage = ask(c1_messages)
print("\nANSWER:\n" + c1_answer)
print("\nToken accounting (o200k_base):")
print("  system:", system_tokens, "question:", question_tokens, "context:", context_tokens,
      "total:", system_tokens + question_tokens + context_tokens)
print("  API usage.prompt_tokens:", c1_usage)
print("  chunks:", len(c1_results), "tokens per chunk:", chunk_tokens,
      "context characters:", len(context_only))
print("The API total also includes chat-message serialization/role tokens and provider-side formatting, so it need not equal a plain concatenation encoded by tiktoken.")
print("Retrieval literally adds the selected chunk IDs, titles, and chunk text to the user message; the model does not receive the corpus itself.")

SYSTEM MESSAGE:
Answer the question using only the provided context.
If the answer cannot be found in the context, say that you do not know.
Cite the context blocks you used, like [1] or [2][3].

CONTEXT AND QUESTION:
Context:
[1] GO-04 — How much the grant pays
The grant amount depends only on the income band, not on the GPA. A student in band 1 receives 250,000 tenge for the academic year. A student in band 2 receives 150,000 tenge for the academic year. The amount is paid in two equal instalments, the first in October and the second in February. There is no partial grant: a student either receives the full amount for their band or nothing.

[2] GO-03 — Income bands
The income band is set from the monthly household income per person, declared on form GR-210. Band 1: below 85,000 tenge per person per month. Band 2: from 85,000 up to 150,000 tenge per person per month. Band 3: above 150,000 tenge per person per month. Households in band 3 are not eligible for the need-based grant, alth


ANSWER:
A student in income band 2 receives **150,000 tenge for the academic year**, paid in **two equal instalments of 75,000 tenge**: one in **October** and one in **February**. [1]

Token accounting (o200k_base):
  system: 44 question: 18 context: 314 total: 376
  API usage.prompt_tokens: 390
  chunks: 3 tokens per chunk: [90, 104, 85] context characters: 1221
The API total also includes chat-message serialization/role tokens and provider-side formatting, so it need not equal a plain concatenation encoded by tiktoken.
Retrieval literally adds the selected chunk IDs, titles, and chunk text to the user message; the model does not receive the corpus itself.


## C2 — three systems, six questions

| Question | LLM only | BM25 RAG | Embedding RAG |
|---|---|---|---|
| C-01 | ✗ asked for scheme | ✓ 150,000; Oct/Feb | ✓ 150,000; Oct/Feb |
| C-02 | ✗ generic advice | ✓ 14 days; GR-305 | ✓ 14 days; GR-305 |
| C-03 | ✗ no figure | ✓ 2.67 / 4.0 | ✓ 2.67 / 4.0 |
| C-04 | ✓ October 2017; 2031 | ✓ October 2017; 2031 | ✓ October 2017; 2031 |
| C-05 | ✓ 1963; Almaty Institute | ✓ 1963; Almaty Institute | ✓ 1963; Almaty Institute |
| C-06 | ✗ invented US funding details | ✓ correctly says unknown | ✓ correctly says unknown |
| **agrees with reference** | **2/6** | **6/6** | **6/6** |

The complete, unabridged answers and references are printed by the executed cell below. C-04 and C-05 illustrate that an LLM can know common Kazakhstan facts without retrieval, while C-01–C-03 and especially the fictional handbook question C-06 show why grounding changes the result.


In [9]:
systems = {"LLM only": [], "BM25 RAG": [], "Embedding RAG": []}
for row in qa:
    question = row["question"]
    plain, _ = ask([{"role": "user", "content": question}])
    bm_answer, _ = ask(rag_messages(question, top_bm25(question)))
    em_answer, _ = ask(rag_messages(question, top_embed(question)))
    systems["LLM only"].append(plain)
    systems["BM25 RAG"].append(bm_answer)
    systems["Embedding RAG"].append(em_answer)
    print("\n" + "=" * 96)
    print(row["id"], question)
    print("Reference:", row["reference"])
    print("LLM only:", plain)
    print("BM25 RAG:", bm_answer)
    print("Embedding RAG:", em_answer)
    time.sleep(0.2)


C-01 How much does a student in income band 2 receive, and when is it paid?
Reference: 150,000 tenge for the academic year, in two equal instalments: October and February.
LLM only: Which student-support scheme and country do you mean? “Income band 2” is used by several programs, and the amount and payment date/frequency vary. Please share the scheme name, institution, or a link.
BM25 RAG: A student in income band 2 receives **150,000 tenge for the academic year**, paid in **two equal instalments**: the first in **October** and the second in **February**. [1]
Embedding RAG: A student in income band 2 receives **150,000 tenge for the academic year**, paid in **two equal instalments of 75,000 tenge**: the first in **October** and the second in **February**. [1]



C-02 My grant application was refused. How long do I have to appeal, and which form do I use?
Reference: 14 calendar days from the date on the decision letter, using form GR-305.
LLM only: The appeal deadline and form depend on the **grant scheme and jurisdiction**—there is no universal rule. They are usually stated in the refusal letter or decision notice under headings such as **“Appeal,” “Review,”** or **“Reconsideration.”**

Please tell me:

- the **name of the grant/programme**
- the **country or agency** that decided it
- the **date on the refusal letter** (and when you received it)

You can also paste the relevant part of the refusal notice, with personal details removed. Check it promptly, because deadlines may run from the **date of the decision or the date you received it**, and missing the deadline can prevent an appeal.
BM25 RAG: You have **14 calendar days from the date on the decision letter** to appeal. Use **Form GR-305**. [3]
Embedding RAG: You have **14 calendar days


C-03 What is the minimum GPA needed to qualify for the need-based grant?
Reference: A cumulative GPA of at least 2.67 on the 4.0 scale. (The 3.0 in GO-05 is the archived 2025 rule.)
LLM only: The minimum GPA depends on the specific need-based grant and institution. Which grant or school are you referring to?
BM25 RAG: The minimum GPA is **2.67 on a 4.0 scale**. [2]
Embedding RAG: The minimum GPA needed is **2.67 on a 4.0 scale**. [1]



C-04 When did Kazakhstan decide to switch the Kazakh alphabet from Cyrillic to Latin, and when is the switch now due to finish?
Reference: A presidential decree of October 2017 ordered the change for 2025; in January 2021 the target was moved to 2031.
LLM only: Kazakhstan formally decided to change the Kazakh alphabet from Cyrillic to Latin script on **26 October 2017**, when President Nursultan Nazarbayev signed the relevant decree. The initial target was to complete the change by **2025**.

That deadline was later revised. Under the current phased timetable, the transition is expected to be completed by **2031**.
BM25 RAG: Kazakhstan decided to switch from Cyrillic to Latin in **October 2017**. The transition was initially planned to finish by 2025, but the target was later postponed; it is now due to finish in **2031**. [1][2]
Embedding RAG: Kazakhstan decided to switch from Cyrillic to Latin script in **October 2017**. The transition is now due to be completed by **2031**. [1][2]



C-05 When was Narxoz University founded, and what was it called at the start?
Reference: In 1963, as the Almaty Institute of National Economy; it took its current name in 2016.
LLM only: Narxoz University was founded in **1963** as the **Alma-Ata Institute of National Economy**.
BM25 RAG: Narxoz University was founded in 1963 as the Almaty Institute of National Economy. [1]
Embedding RAG: Narxoz University was founded in 1963 as the **Almaty Institute of National Economy**. [1]



C-06 I am a master's student. How much need-based grant can I get?
Reference: The corpus does not say. GO-20 says master's students have a separate scheme that the handbook does not describe.
LLM only: It depends heavily on your country, school, program, enrollment status, and financial circumstances.

**If you’re in the U.S.:**

- **Federal Pell Grant:** Usually **not available for master’s students**; it is primarily for undergraduates who have not earned a bachelor’s degree.
- **Federal TEACH Grant:** Graduate students may receive up to **$4,000 per year** if they meet the eligibility requirements and commit to teaching in a high-need field/school.
- **School- or state-based grants:** These vary widely—some master’s students receive a few hundred dollars, while others may receive several thousand or full tuition support.
- **Need-based financial aid:** Graduate students are generally more likely to qualify for **federal Direct Unsubsidized Loans** and possibly **Grad PLUS Loans**, 

## C3 — three questions taken apart

**C-03 (minimum GPA).** Both retrievers included GO-02, the active 2026 qualification rule, and both generated the correct 2.67 answer. This is grounded and specific. The archived 3.0 rule is GO-05; it appears in the embedding top 3 but the model did not repeat it. The fact was present, so an answer that chose 3.0 would have been a generation failure, not retrieval failure.

**C-06 (master’s grant).** Both retrievers included GO-20; EmbeddingGemma placed it first. GO-20 explicitly says that the separate master’s scheme is not described. The two RAG answers correctly say “I don’t know”; the LLM-only system instead invented United States funding programmes. Retrieval therefore made the answer more grounded and prevented hallucination.

**C-02 (appeal).** Both retrieve the answer-bearing appeal chunks. BM25 also ranks the archived 2025 scheme first because `decision`/`grant` overlap, which was a potential failure: the required fact is nevertheless present in GO-08 and GO-12, and the model answered from those blocks. This is a useful near-miss: had it followed the archived chunk, generation would have failed despite successful retrieval.

The executed cells below print the IDs, scores, complete contexts, answers, and an explicit present/absent diagnosis for each case.


In [10]:
def dissect(qid):
    idx = next(i for i, row in enumerate(qa) if row["id"] == qid)
    row = qa[idx]
    b, e = top_bm25(row["question"]), top_embed(row["question"])
    print("QUESTION", qid, ":", row["question"])
    print("REFERENCE:", row["reference"], "| reference chunks:", row["relevant"])
    print("\nBM25 retrieved:"); show(b)
    print("\nEmbeddingGemma retrieved:"); show(e)
    print("\nBM25 context sent:\n", context_text(b))
    print("\nEmbedding context sent:\n", context_text(e))
    print("\nAnswers — LLM only:", systems["LLM only"][idx])
    print("BM25 RAG:", systems["BM25 RAG"][idx])
    print("Embedding RAG:", systems["Embedding RAG"][idx])
    for label, results in [("BM25", b), ("Embedding", e)]:
        ids = {item["id"] for item, _ in results}
        verdict = "reference fact was present: any omission is a generation failure" if ids & set(row["relevant"]) else "reference fact was absent: this is a retrieval failure"
        print(f"Diagnosis for {label}: {verdict}.")

dissect("C-03")

QUESTION C-03 : What is the minimum GPA needed to qualify for the need-based grant?
REFERENCE: A cumulative GPA of at least 2.67 on the 4.0 scale. (The 3.0 in GO-05 is the archived 2025 rule.) | reference chunks: ['GO-02']

BM25 retrieved:
1. GO-16 | 15.7985 | Combining the grant with other support
   The need-based grant can be held together with a merit scholarship, but not together with a second need-based grant from any other source. The total o
2. GO-02 | 14.0587 | Who qualifies
   An applicant qualifies for the 2026 grant when all three conditions hold at the time of the decision: their cumulative GPA is at least 2.67 on the 4.0
3. GO-06 | 13.8516 | Form GR-104: the application form
   Form GR-104 is the application form for the need-based study grant. It is the only way to apply; an email or a letter is not an application. GR-104 is

EmbeddingGemma retrieved:
1. GO-02 | 0.6286 | Who qualifies
   An applicant qualifies for the 2026 grant when all three conditions hold at the time

In [11]:
dissect('C-06')

QUESTION C-06 : I am a master's student. How much need-based grant can I get?
REFERENCE: The corpus does not say. GO-20 says master's students have a separate scheme that the handbook does not describe. | reference chunks: ['GO-20']

BM25 retrieved:
1. GO-16 | 16.9958 | Combining the grant with other support
   The need-based grant can be held together with a merit scholarship, but not together with a second need-based grant from any other source. The total o
2. GO-01 | 14.1725 | About the need-based study grant 2026
   The need-based study grant 2026 is run by the Grant Office for full-time undergraduate students. It exists so that a student from a low-income househo
3. GO-20 | 13.7202 | Who cannot apply
   The need-based grant is for full-time undergraduate students only. Part-time students, distance-learning students and incoming exchange students canno

EmbeddingGemma retrieved:
1. GO-20 | 0.6664 | Who cannot apply
   The need-based grant is for full-time undergraduate students onl

In [12]:
dissect('C-02')

QUESTION C-02 : My grant application was refused. How long do I have to appeal, and which form do I use?
REFERENCE: 14 calendar days from the date on the decision letter, using form GR-305. | reference chunks: ['GO-08', 'GO-12', 'GO-23']

BM25 retrieved:
1. GO-05 | 17.2414 | Archived: the 2025 scheme (no longer in force)
   ARCHIVED — kept for reference only. Under the 2025 scheme the minimum GPA for the grant was 3.0, band 1 received 200,000 tenge, band 2 received 120,00
2. GO-08 | 14.5287 | Form GR-305: appeals
   Form GR-305 is used to appeal a decision of the Grant Office. It is the appeal form and nothing else: it cannot be used to apply for the first time or
3. GO-12 | 13.5735 | How to appeal a decision
   A student who disagrees with a decision may appeal within 14 calendar days of the date on the decision letter, using form GR-305. The appeal is read b

EmbeddingGemma retrieved:
1. GO-08 | 0.6029 | Form GR-305: appeals
   Form GR-305 is used to appeal a decision of the Grant Of

---
# Final questions

1. **Main difference.** BM25 scores sparse token overlap with term-frequency and inverse-document-frequency weighting. Embedding retrieval compares dense semantic vectors, so it can rank related wording without exact overlap.

2. **When can BM25 do better?** It can win when the question includes a rare exact identifier, title, number, or form code, such as `GR-305`; an embedding can blur that exact distinction.

3. **When can embeddings do better?** They help with paraphrases, synonyms, morphology, and cross-language semantic matches where BM25 has few or no literal matching tokens.

4. **Why cosine?** Cosine measures the angle between embedding vectors, emphasizing direction/semantic relationship instead of vector length, which makes independently encoded texts comparable.

5. **What is sent to an LLM?** The system instructions, question, and only the selected context blocks (their IDs/titles/text) are serialized into the request, not the original collection.

6. **Does the LLM search the original documents?** No. Retrieval happens in the application before the call; the LLM only continues the prompt it receives.

7. **What if retrieval is irrelevant?** The model may answer from irrelevant evidence, become less grounded, or correctly say it does not know. If the needed fact never reached the prompt, that is a retrieval failure.

In [13]:
print("K comparison on C-03 with EmbeddingGemma retrieval")
for k in (1, 3, 8):
    results = top_embed(qa[2]["question"], k=k)
    answer, _ = ask(rag_messages(qa[2]["question"], results))
    print(f"\nK={k}; ids={[item['id'] for item, _ in results]}; context tokens={len(enc.encode(context_text(results)))}")
    print(answer)

print("\n8. Increasing K can improve recall by including a missing relevant chunk, but it can also add contradictory or distracting chunks, dilute instructions, and consume context. The K=1/3/8 outputs above show the actual trade-off on C-03.")
print("\n9. A vector database is not required for a 74-chunk corpus: a NumPy similarity matrix is sufficient. A vector database becomes useful for persistence, filtering, and large-scale approximate nearest-neighbour search.")
print(f"\n10. Over the ten queries, BM25 hit@3={b_total}/10 and EmbeddingGemma hit@3={e_total}/10. Compare these with the six C2 answer sets above; retrieval quality and answer correctness are related but not identical because a model can still mishandle supplied context.")

K comparison on C-03 with EmbeddingGemma retrieval



K=1; ids=['GO-02']; context tokens=112
The minimum GPA needed is **2.67 on a 4.0 scale**. [1]



K=3; ids=['GO-02', 'GO-20', 'GO-05']; context tokens=286
The minimum GPA needed is **2.67 on a 4.0 scale** for the 2026 grant. [1]



K=8; ids=['GO-02', 'GO-20', 'GO-05', 'GO-16', 'GO-01', 'GO-06', 'GO-19', 'GO-17']; context tokens=776
The minimum GPA needed to qualify for the 2026 need-based grant is **2.67 on a 4.0 scale**. [1]

8. Increasing K can improve recall by including a missing relevant chunk, but it can also add contradictory or distracting chunks, dilute instructions, and consume context. The K=1/3/8 outputs above show the actual trade-off on C-03.

9. A vector database is not required for a 74-chunk corpus: a NumPy similarity matrix is sufficient. A vector database becomes useful for persistence, filtering, and large-scale approximate nearest-neighbour search.

10. Over the ten queries, BM25 hit@3=6/10 and EmbeddingGemma hit@3=9/10. Compare these with the six C2 answer sets above; retrieval quality and answer correctness are related but not identical because a model can still mishandle supplied context.


## Conclusion

This notebook makes the RAG boundary visible: the retrieval lists, contexts, token accounting, and answers let a retrieval miss be separated from a generation miss.